In [ ]:
from google.colab import files
uploaded = files.upload()

Saving expanded_music_recommendation_data.csv to expanded_music_recommendation_data.csv


In [ ]:
!pip install chardet


In [ ]:
import chardet

# Detect encoding
with open('expanded_music_recommendation_data.csv', 'rb') as f:
    result = chardet.detect(f.read(10000))  # Read the first 10,000 bytes
    print(result)


{'encoding': 'ISO-8859-1', 'confidence': 0.73, 'language': ''}


In [ ]:
categorical_columns = ['weather_condition', 'time_of_day', 'day_of_week']


In [ ]:
categorical_columns = ['weather_condition', 'time_of_day']


In [ ]:
print(df.columns.tolist())



['User ID', 'Location', 'Weather Condition', 'Temperature (°C)', 'Humidity (%)', 'Time of Day', 'Day of the Week', 'Season', 'Preferred Genre', 'Previous Song Ratings', 'Mood Tag', 'Song ID / Name', 'Feedback Score']


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Step 1: Load Dataset with correct encoding
df = pd.read_csv('expanded_music_recommendation_data.csv', encoding='ISO-8859-1')

# Step 2: Encode Categorical Data
# Use the exact column names from your dataset
categorical_columns = ['Weather Condition', 'Time of Day', 'Day of the Week', 'Season', 'Preferred Genre', 'Mood Tag']
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

# Step 3: Feature Scaling
# Columns to scale
numerical_columns = ['Temperature (°C)', 'Humidity (%)', 'Previous Song Ratings', 'Feedback Score']
scaler = StandardScaler()
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

# The dataframe `df` is now preprocessed and ready for further steps
print(df.head())


   User ID     Location  Temperature (°C)  Humidity (%)  \
0       92     Warangal          0.274005     -1.346959   
1       38    Hyderabad         -0.745143     -1.157059   
2       33   Patancheru          1.293153     -1.584333   
3       73   Karimnagar          1.293153     -1.299484   
4       41  Lingampally          0.637986     -1.157059   

   Previous Song Ratings Song ID / Name  Feedback Score  \
0              -0.738776     Rowdy Baby       -1.267566   
1               0.624280      Despacito       -0.918374   
2              -0.057248  Enjoy Enjaami        1.525968   
3              -0.738776   Shape of You        0.827584   
4              -0.738776     Rowdy Baby       -1.616758   

   Weather Condition_Cloudy  Weather Condition_Rainy  Weather Condition_Snowy  \
0                     False                    False                    False   
1                      True                    False                    False   
2                     False                    

In [ ]:
# Step 4: Save the preprocessed dataframe to a new CSV file
df.to_csv('preprocessed_music_recommendation_data.csv', index=False)


In [ ]:
# Save the preprocessed dataframe to a CSV file
df.to_csv('/content/preprocessed_music_recommendation_data.csv', index=False)


In [ ]:
import pandas as pd

# Aggregate duplicate entries by taking the mean of the Feedback Score
df_agg = df.groupby(['User ID', 'Song ID / Name'], as_index=False)['Feedback Score'].mean()

# Now pivot the DataFrame to create the user-song interaction matrix
user_song_matrix = df_agg.pivot(index='User ID', columns='Song ID / Name', values='Feedback Score')

# Fill missing values with 0 (assuming the user hasn't rated a song)
user_song_matrix = user_song_matrix.fillna(0)


In [ ]:
# Apply Singular Value Decomposition (SVD)
svd = TruncatedSVD(n_components=20)  # Set n_components <= number of features (20)
svd_matrix = svd.fit_transform(user_song_matrix)

# Reconstruct the matrix with the approximated values
predicted_ratings = svd.inverse_transform(svd_matrix)

# Convert the predicted ratings back to a DataFrame
predicted_ratings_df = pd.DataFrame(predicted_ratings, columns=user_song_matrix.columns)


In [ ]:
def recommend_songs(user_id, predicted_ratings_df, top_n=5):
    user_ratings = predicted_ratings_df.iloc[user_id - 1]
    recommendations = user_ratings[user_ratings > 0].sort_values(ascending=False).head(top_n)
    return recommendations.index.tolist()

# Example: Recommend 5 songs for User 1
recommend_songs(1, predicted_ratings_df)


['Cheap Thrills', 'O Cheliya', 'Ramuloo Ramulaa', 'Leharaayi', 'Dance Monkey']

In [ ]:
# Print the column names to verify
print(df.columns)


Index(['User ID', 'Location', 'Temperature (°C)', 'Humidity (%)',
       'Previous Song Ratings', 'Song ID / Name', 'Feedback Score',
       'Weather Condition_Cloudy', 'Weather Condition_Rainy',
       'Weather Condition_Snowy', 'Weather Condition_Windy',
       'Time of Day_Evening', 'Time of Day_Morning', 'Time of Day_Night',
       'Day of the Week_Monday', 'Day of the Week_Saturday',
       'Day of the Week_Sunday', 'Day of the Week_Thursday',
       'Day of the Week_Tuesday', 'Day of the Week_Wednesday', 'Season_Spring',
       'Season_Summer', 'Season_Winter', 'Preferred Genre_Country',
       'Preferred Genre_Electronic', 'Preferred Genre_Hip-Hop',
       'Preferred Genre_Jazz', 'Preferred Genre_Pop', 'Preferred Genre_R&B',
       'Preferred Genre_Reggae', 'Preferred Genre_Rock', 'Mood Tag_Focus',
       'Mood Tag_Happy', 'Mood Tag_Relaxing', 'Mood Tag_Romantic',
       'Mood Tag_Sad'],
      dtype='object')


In [ ]:
# Create a new column for the preferred genre based on the highest value in the genre columns
df['Preferred Genre'] = df[['Preferred Genre_Country', 'Preferred Genre_Electronic', 'Preferred Genre_Hip-Hop',
                             'Preferred Genre_Jazz', 'Preferred Genre_Pop', 'Preferred Genre_R&B',
                             'Preferred Genre_Reggae', 'Preferred Genre_Rock']].idxmax(axis=1)

# This will create a new column 'Preferred Genre' with the name of the genre that has the highest value (1)


In [ ]:
# Combine song features into a single column (one-hot encoded genres, mood, and season)
df['song_features'] = df[['Preferred Genre_Country', 'Preferred Genre_Electronic', 'Preferred Genre_Hip-Hop',
                           'Preferred Genre_Jazz', 'Preferred Genre_Pop', 'Preferred Genre_R&B',
                           'Preferred Genre_Reggae', 'Preferred Genre_Rock',
                           'Mood Tag_Focus', 'Mood Tag_Happy', 'Mood Tag_Relaxing', 'Mood Tag_Romantic', 'Mood Tag_Sad',
                           'Season_Spring', 'Season_Summer', 'Season_Winter']].apply(lambda row: ' '.join(row.astype(str)), axis=1)

# Vectorize the features using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
song_features_matrix = vectorizer.fit_transform(df['song_features'])

# Compute cosine similarity between songs
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(song_features_matrix, song_features_matrix)

# Print the cosine similarity matrix
print(cosine_sim)


[[1.         1.         0.9963846  ... 1.         1.         0.98714178]
 [1.         1.         0.9963846  ... 1.         1.         0.98714178]
 [0.9963846  0.9963846  1.         ... 0.9963846  0.9963846  0.99715304]
 ...
 [1.         1.         0.9963846  ... 1.         1.         0.98714178]
 [1.         1.         0.9963846  ... 1.         1.         0.98714178]
 [0.98714178 0.98714178 0.99715304 ... 0.98714178 0.98714178 1.        ]]


In [ ]:
print(df['Song ID / Name'].unique())  # This will print all unique song names in the DataFrame


['Rowdy Baby' 'Despacito' 'Enjoy Enjaami' 'Shape of You' 'Seeti Maar'
 'Dance Monkey' 'Cheap Thrills' 'O Cheliya' 'Pilla Ra' 'Stay' 'Rockstar'
 'Butta Bomma' 'Closer' 'Ramuloo Ramulaa' 'Leharaayi' 'Perfect'
 'Samajavaragamana' 'Nee Kannu Neeli Samudram' 'Blinding Lights'
 'Heat Waves']


In [ ]:
# Clean up the song names by stripping extra spaces
df['Song ID / Name'] = df['Song ID / Name'].str.strip()

# List all unique song names again
print(df['Song ID / Name'].unique())


['Rowdy Baby' 'Despacito' 'Enjoy Enjaami' 'Shape of You' 'Seeti Maar'
 'Dance Monkey' 'Cheap Thrills' 'O Cheliya' 'Pilla Ra' 'Stay' 'Rockstar'
 'Butta Bomma' 'Closer' 'Ramuloo Ramulaa' 'Leharaayi' 'Perfect'
 'Samajavaragamana' 'Nee Kannu Neeli Samudram' 'Blinding Lights'
 'Heat Waves']


In [ ]:
# Check if there are any missing or NaN values in the 'Song ID / Name' column
print(df['Song ID / Name'].isnull().sum())


0


In [ ]:
# Strip leading and trailing spaces in song names
df['Song ID / Name'] = df['Song ID / Name'].str.strip()

# Check the first few rows to ensure no extra spaces
print(df[['Song ID / Name']].head())


  Song ID / Name
0     Rowdy Baby
1      Despacito
2  Enjoy Enjaami
3   Shape of You
4     Rowdy Baby


In [ ]:
# Print all unique song names
print("List of unique song names:")
print(df['Song ID / Name'].unique())

# Manually pick a song from the printed list
search_song = 'Song A'  # Replace with a song from the printed list

# Check if the song exists in the dataset
if search_song in df['Song ID / Name'].values:
    print(f"Song '{search_song}' found in dataset.")
else:
    print(f"Song '{search_song}' not found in dataset.")


List of unique song names:
['Rowdy Baby' 'Despacito' 'Enjoy Enjaami' 'Shape of You' 'Seeti Maar'
 'Dance Monkey' 'Cheap Thrills' 'O Cheliya' 'Pilla Ra' 'Stay' 'Rockstar'
 'Butta Bomma' 'Closer' 'Ramuloo Ramulaa' 'Leharaayi' 'Perfect'
 'Samajavaragamana' 'Nee Kannu Neeli Samudram' 'Blinding Lights'
 'Heat Waves']
Song 'Song A' not found in dataset.


In [ ]:
def recommend_songs_content(song_id, cosine_sim, top_n=5):
    # Ensure the song_id is cleaned and matches in the dataset
    song_id = song_id.strip()

    # Check if song_id exists in the dataset
    if song_id not in df['Song ID / Name'].values:
        return f"Song '{song_id}' not found in the dataset."

    # Get the index of the song in the DataFrame
    song_idx = df[df['Song ID / Name'] == song_id].index[0]

    # Get similarity scores for the song
    similarity_scores = list(enumerate(cosine_sim[song_idx]))

    # Sort the songs based on similarity scores (highest first)
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # Get the indices of the top N similar songs
    recommended_song_indices = [x[0] for x in similarity_scores[1:top_n+1]]

    # Return the song names of the top N similar songs
    return df.iloc[recommended_song_indices]['Song ID / Name'].tolist()

# Use an actual song name that exists in the dataset
valid_song_name = 'Rowdy Baby'  # Replace with a valid song from the unique list
recommended_songs = recommend_songs_content(valid_song_name, cosine_sim)
print("Recommended Songs:")
print(recommended_songs)


Recommended Songs:
['Despacito', 'Shape of You', 'Despacito', 'Despacito', 'Cheap Thrills']


In [ ]:
# Sample list of recommendations (replace with actual recommended songs from your model)
recommendations = [
    {'Song ID / Name': 'Rowdy Baby', 'Preferred Genre': ['Classical']},
    {'Song ID / Name': 'Despacito', 'Preferred Genre': ['Pop']},
    {'Song ID / Name': 'Enjoy Enjaami', 'Preferred Genre': ['Electronic']},
    {'Song ID / Name': 'Shape of You', 'Preferred Genre': ['Pop']},
    {'Song ID / Name': 'Blinding Lights', 'Preferred Genre': ['Dance']}
]

def adjust_recommendations_based_on_context(weather, recommendations):
    if weather == 'Rainy':
        calm_genres = ['Classical', 'Jazz', 'Blues']
        recommendations = [song for song in recommendations if any(genre in calm_genres for genre in song['Preferred Genre'])]
    elif weather == 'Sunny':
        upbeat_genres = ['Pop', 'Dance', 'Electronic']
        recommendations = [song for song in recommendations if any(genre in upbeat_genres for genre in song['Preferred Genre'])]
    return recommendations

# Example: Adjust recommendations for rainy weather
adjusted_recommendations = adjust_recommendations_based_on_context('Rainy', recommendations)

# Print adjusted recommendations
print(adjusted_recommendations)


[{'Song ID / Name': 'Rowdy Baby', 'Preferred Genre': ['Classical']}]


In [ ]:
song_genre_mapping = {
    'Rowdy Baby': 'Pop',
    'Despacito': 'Pop',
    'Enjoy Enjaami': 'Folk',
    'Shape of You': 'Pop',
    'Seeti Maar': 'Bollywood',
    'Dance Monkey': 'Pop',
    'Cheap Thrills': 'Pop',
    'O Cheliya': 'Romantic',
    'Pilla Ra': 'Folk',
    'Stay': 'Pop',
    'Rockstar': 'Bollywood',
    'Butta Bomma': 'Pop',
    'Closer': 'Pop',
    'Ramuloo Ramulaa': 'Bollywood',
    'Leharaayi': 'Romantic',
    'Perfect': 'Pop',
    'Samajavaragamana': 'Pop',
    'Nee Kannu Neeli Samudram': 'Romantic',
    'Blinding Lights': 'Pop',
    'Heat Waves': 'Pop'
}


In [ ]:
# Example call for the functions
collaborative_recommendations = recommend_songs(user_id=1, predicted_ratings_df=predicted_ratings_df, top_n=5)
content_recommendations = recommend_songs_content(song_id='Rowdy Baby', cosine_sim=cosine_sim, top_n=5)

# Now print the recommendations
print("Recommended Songs from Collaborative Filtering:", collaborative_recommendations)
print("Recommended Songs from Content-based Filtering:", content_recommendations)


Recommended Songs from Collaborative Filtering: ['Rowdy Baby', 'Despacito', 'Enjoy Enjaami', 'Shape of You', 'Seeti Maar']
Recommended Songs from Content-based Filtering: ['Despacito', 'Blinding Lights', 'Heat Waves', 'Shape of You', 'Dance Monkey']


In [ ]:
# Debug: Print all the song names in the dataset
print("Song IDs in the dataset:", df['Song ID / Name'].unique())

# Check your recommendations to make sure they match
print("Recommended Songs from Collaborative Filtering:", collaborative_recommendations)
print("Recommended Songs from Content-based Filtering:", content_recommendations)


Song IDs in the dataset: ['Rowdy Baby' 'Despacito' 'Enjoy Enjaami' 'Shape of You' 'Seeti Maar'
 'Dance Monkey' 'Cheap Thrills' 'O Cheliya' 'Pilla Ra' 'Stay' 'Rockstar'
 'Butta Bomma' 'Closer' 'Ramuloo Ramulaa' 'Leharaayi' 'Perfect'
 'Samajavaragamana' 'Nee Kannu Neeli Samudram' 'Blinding Lights'
 'Heat Waves']
Recommended Songs from Collaborative Filtering: ['Rowdy Baby', 'Despacito', 'Enjoy Enjaami', 'Shape of You', 'Seeti Maar']
Recommended Songs from Content-based Filtering: ['Despacito', 'Blinding Lights', 'Heat Waves', 'Shape of You', 'Dance Monkey']


In [ ]:
def adjust_recommendations_based_on_context(weather, recommendations):
    calm_genres = ['Classical', 'Jazz', 'Blues']
    upbeat_genres = ['Pop', 'Dance', 'Electronic']

    # Adjust recommendations based on weather
    if weather == 'Rainy':
        recommendations = [song for song in recommendations if song_genre_mapping.get(song, '') in calm_genres]
    elif weather == 'Sunny':
        recommendations = [song for song in recommendations if song_genre_mapping.get(song, '') in upbeat_genres]
    return recommendations


In [ ]:
def hybrid_recommendation(user_id, song_id, predicted_ratings_df, cosine_sim, weather, top_n=5):
    # Get collaborative and content-based recommendations
    collaborative_recommendations = recommend_songs(user_id, predicted_ratings_df, top_n)
    content_recommendations = recommend_songs_content(song_id, cosine_sim, top_n)

    # Debugging: Check what's being returned by the recommendation functions
    print("Collaborative Recommendations:", collaborative_recommendations)
    print("Content-based Recommendations:", content_recommendations)

    # Combine the two recommendation lists
    combined_recommendations = list(set(collaborative_recommendations + content_recommendations))

    # Adjust recommendations based on contextual information (weather)
    adjusted_recommendations = adjust_recommendations_based_on_context(weather, combined_recommendations)

    # Debugging: Check combined and adjusted recommendations
    print("Combined Recommendations:", combined_recommendations)
    print("Adjusted Recommendations:", adjusted_recommendations)

    return adjusted_recommendations


In [ ]:
# Check the hybrid recommendation for a user and song with valid data
recommendations = hybrid_recommendation(user_id=1, song_id='Rowdy Baby', predicted_ratings_df=predicted_ratings_df, cosine_sim=cosine_sim, weather='Rainy', top_n=5)
print("Final Recommendations:", recommendations)


Collaborative Recommendations: ['Rowdy Baby', 'Despacito', 'Enjoy Enjaami', 'Shape of You', 'Seeti Maar']
Content-based Recommendations: ['Despacito', 'Blinding Lights', 'Heat Waves', 'Shape of You', 'Dance Monkey']
Combined Recommendations: ['Dance Monkey', 'Despacito', 'Shape of You', 'Rowdy Baby', 'Enjoy Enjaami', 'Heat Waves', 'Seeti Maar', 'Blinding Lights']
Adjusted Recommendations: []
Final Recommendations: []


In [ ]:
# Check what genre each song is mapped to
for song in ['Rowdy Baby', 'Despacito', 'Shape of You']:  # Add a few examples
    print(f"{song}: {song_genre_mapping.get(song, 'No Genre Found')}")


Rowdy Baby: Pop
Despacito: Pop
Shape of You: Pop


In [ ]:
def adjust_recommendations_based_on_context(weather, recommendations):
    print("Recommendations before adjusting based on context:", recommendations)  # Debug print
    # Adjust recommendations based on weather
    if weather == 'Rainy':
        calm_genres = ['Classical', 'Jazz', 'Blues']
        recommendations = [song for song in recommendations if any(genre in calm_genres for genre in song['Preferred Genre'])]
    elif weather == 'Sunny':
        upbeat_genres = ['Pop', 'Dance', 'Electronic']
        recommendations = [song for song in recommendations if any(genre in upbeat_genres for genre in song['Preferred Genre'])]

    print("Recommendations after adjusting based on context:", recommendations)  # Debug print
    return recommendations

def hybrid_recommendation(user_id, song_id, predicted_ratings_df, cosine_sim, weather, humidity, top_n=5):
    # Get collaborative and content-based recommendations
    collaborative_recommendations = recommend_songs(user_id, predicted_ratings_df, top_n)
    content_recommendations = recommend_songs_content(song_id, cosine_sim, top_n)

    print("Collaborative Recommendations:", collaborative_recommendations)  # Debug print
    print("Content-Based Recommendations:", content_recommendations)  # Debug print

    # Combine the two recommendation lists
    combined_recommendations = list(set(collaborative_recommendations + content_recommendations))
    print("Combined Recommendations:", combined_recommendations)  # Debug print

    # Ensure each recommendation includes song details (e.g., song name and genre)
    song_details = get_song_details(combined_recommendations)  # This function should return the song details
    print("Song Details:", song_details)  # Debug print

    # Adjust recommendations based on contextual information (weather, etc.)
    adjusted_recommendations = adjust_recommendations_based_on_context(weather, song_details)

    # Further adjustments can be made based on humidity, time of day, etc.
    if humidity > 80:
        # Example: If humidity is high, recommend relaxing songs
        relaxing_genres = ['Ambient', 'Classical', 'Jazz']
        adjusted_recommendations = [song for song in adjusted_recommendations if any(genre in relaxing_genres for genre in song['Preferred Genre'])]

    return adjusted_recommendations

# Function to get song details based on song name (just an example)
def get_song_details(song_names):
    # Assuming you have a DataFrame or a dictionary with song details
    song_details_df = pd.DataFrame({
        'Song ID / Name': ['Rowdy Baby', 'Despacito', 'Enjoy Enjaami', 'Shape of You'],
        'Preferred Genre': ['Pop', 'Reggae', 'Rock', 'Pop'],
        'Mood Tag': ['Happy', 'Relaxing', 'Romantic', 'Happy'],
        # Add more details as needed (e.g., 'Season', 'Weather Condition')
    })

    # Ensure the song names exist in your dataset and return the details
    song_details = song_details_df[song_details_df['Song ID / Name'].isin(song_names)]
    print("Retrieved Song Details:", song_details)  # Debug print
    return song_details.to_dict(orient='records')

# Example: Hybrid recommendation for User 1 and Song 'Rowdy Baby' in rainy weather with high humidity
recommendations = hybrid_recommendation(1, 'Rowdy Baby', predicted_ratings_df, cosine_sim, 'Rainy', humidity=85, top_n=5)
print("Final Recommendations:", recommendations)  # Final output


Collaborative Recommendations: ['Rowdy Baby', 'Despacito', 'Enjoy Enjaami', 'Shape of You', 'Seeti Maar']
Content-Based Recommendations: ['Despacito', 'Blinding Lights', 'Heat Waves', 'Shape of You', 'Dance Monkey']
Combined Recommendations: ['Dance Monkey', 'Despacito', 'Shape of You', 'Rowdy Baby', 'Enjoy Enjaami', 'Heat Waves', 'Seeti Maar', 'Blinding Lights']
Retrieved Song Details:   Song ID / Name Preferred Genre  Mood Tag
0     Rowdy Baby             Pop     Happy
1      Despacito          Reggae  Relaxing
2  Enjoy Enjaami            Rock  Romantic
3   Shape of You             Pop     Happy
Song Details: [{'Song ID / Name': 'Rowdy Baby', 'Preferred Genre': 'Pop', 'Mood Tag': 'Happy'}, {'Song ID / Name': 'Despacito', 'Preferred Genre': 'Reggae', 'Mood Tag': 'Relaxing'}, {'Song ID / Name': 'Enjoy Enjaami', 'Preferred Genre': 'Rock', 'Mood Tag': 'Romantic'}, {'Song ID / Name': 'Shape of You', 'Preferred Genre': 'Pop', 'Mood Tag': 'Happy'}]
Recommendations before adjusting based on 

In [3]:
import requests
import pandas as pd

# Sample dataset (replace with actual dataset or DataFrame)
song_data = pd.DataFrame({
    'Song ID / Name': ['Rowdy Baby', 'Despacito', 'Enjoy Enjaami', 'Shape of You', 'Seeti Maar', 'Dance Monkey', 'Cheap Thrills', 'O Cheliya', 'Pilla Ra', 'Stay'],
    'Preferred Genre': ['Pop', 'Reggae', 'Rock', 'Pop', 'Pop', 'Pop', 'Reggae', 'Pop', 'Rock', 'Pop'],
    'Mood Tag': ['Happy', 'Relaxing', 'Romantic', 'Happy', 'Energetic', 'Happy', 'Chill', 'Romantic', 'Upbeat', 'Calm'],
})

# Function to get song details based on song names
def get_song_details(song_names):
    # Assuming the song details are in the song_data DataFrame
    song_details = song_data[song_data['Song ID / Name'].isin(song_names)]
    return song_details.to_dict(orient='records')

# Collaborative Filtering - Example
def recommend_songs(user_id, predicted_ratings_df, top_n=5):
    return ['Rowdy Baby', 'Despacito', 'Shape of You', 'Stay', 'Dance Monkey']

# Content-based Filtering - Example
def recommend_songs_content(song_id, cosine_sim, top_n=5):
    return ['Shape of You', 'Perfect', 'Blinding Lights', 'Closer', 'Heat Waves']

# Adjust recommendations based on weather temperature
def adjust_recommendations_based_on_temperature(temperature, recommendations):
    print(f"Adjusting recommendations based on temperature: {temperature}°C")

    # Define genre preferences based on temperature
    if temperature < 15:
        cozy_genres = ['Classical', 'Jazz', 'Blues']
        recommendations = [song for song in recommendations if song['Preferred Genre'] in cozy_genres]
    elif 15 <= temperature <= 30:
        balanced_genres = ['Pop', 'Rock', 'Dance']
        recommendations = [song for song in recommendations if song['Preferred Genre'] in balanced_genres]
    else:
        energetic_genres = ['Electronic', 'Dance', 'Reggae']
        recommendations = [song for song in recommendations if song['Preferred Genre'] in energetic_genres]

    return recommendations

# Function to get weather data from OpenWeatherMap API
def get_weather_data(location, api_key):
    url = f'http://api.openweathermap.org/data/2.5/weather?q={location}&appid={api_key}&units=metric'
    response = requests.get(url)
    data = response.json()
    if data['cod'] == 200:
        temperature = data['main']['temp']
        weather = data['weather'][0]['description']
        print(f"Weather in {location}: {weather}, Temperature: {temperature}°C")
        return temperature, weather
    else:
        print("Error fetching weather data.")
        return None, None

# Hybrid Recommendation system combining collaborative and content-based filtering
def hybrid_recommendation(user_id, song_id, predicted_ratings_df, cosine_sim, location, api_key, top_n=5):
    # Get current weather data
    temperature, weather = get_weather_data(location, api_key)
    if temperature is None:
        return "Error fetching weather data."

    # Get collaborative and content-based recommendations
    collaborative_recommendations = recommend_songs(user_id, predicted_ratings_df, top_n)
    content_recommendations = recommend_songs_content(song_id, cosine_sim, top_n)

    print(f"Collaborative Recommendations: {collaborative_recommendations}")
    print(f"Content-Based Recommendations: {content_recommendations}")

    # Combine the two recommendation lists
    combined_recommendations = list(set(collaborative_recommendations + content_recommendations))
    print(f"Combined Recommendations (before adjusting for temperature): {combined_recommendations}")

    # Get the song details for the combined recommendations
    song_details = get_song_details(combined_recommendations)
    print(f"Song Details: {song_details}")

    # Adjust recommendations based on temperature
    adjusted_recommendations = adjust_recommendations_based_on_temperature(temperature, song_details)

    return adjusted_recommendations

# Example of using the hybrid recommendation system

# Placeholder for predicted_ratings_df and cosine_sim (usually calculated from data)
predicted_ratings_df = pd.DataFrame()  # This should be your collaborative filtering predictions
cosine_sim = {}  # This should be your cosine similarity matrix (content-based filtering)

# User input for location and weather information
location = input("Enter the location (city name): ")
api_key = "4cee38740c4030f507d5c3c040958919"  # Replace with your actual OpenWeatherMap API key

# Example: Hybrid recommendation for User 1 and Song 'Rowdy Baby' based on real-time weather data
recommendations = hybrid_recommendation(1, 'Rowdy Baby', predicted_ratings_df, cosine_sim, location, api_key, top_n=5)

# Print final recommendations
print("Final Song Recommendations based on weather and temperature:", recommendations)


Enter the location (city name): warangal
Weather in warangal: few clouds, Temperature: 33.64°C
Collaborative Recommendations: ['Rowdy Baby', 'Despacito', 'Shape of You', 'Stay', 'Dance Monkey']
Content-Based Recommendations: ['Shape of You', 'Perfect', 'Blinding Lights', 'Closer', 'Heat Waves']
Combined Recommendations (before adjusting for temperature): ['Shape of You', 'Blinding Lights', 'Perfect', 'Dance Monkey', 'Despacito', 'Closer', 'Rowdy Baby', 'Heat Waves', 'Stay']
Song Details: [{'Song ID / Name': 'Rowdy Baby', 'Preferred Genre': 'Pop', 'Mood Tag': 'Happy'}, {'Song ID / Name': 'Despacito', 'Preferred Genre': 'Reggae', 'Mood Tag': 'Relaxing'}, {'Song ID / Name': 'Shape of You', 'Preferred Genre': 'Pop', 'Mood Tag': 'Happy'}, {'Song ID / Name': 'Dance Monkey', 'Preferred Genre': 'Pop', 'Mood Tag': 'Happy'}, {'Song ID / Name': 'Stay', 'Preferred Genre': 'Pop', 'Mood Tag': 'Calm'}]
Adjusting recommendations based on temperature: 33.64°C
Final Song Recommendations based on weather